# AIC 2026 - Merge Azure Jina embedding parts into a final FAISS index

This notebook verifies completion directly from per-video checkpoints and the Azure
keyframe directory tree. Worker reports are informative but are not required.
The resulting Jina index requires a Jina text-query retriever in the backend.

In [ ]:
!pip install -q azure-storage-blob pyarrow faiss-cpu pandas

In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
import faiss
from azure.storage.blob import BlobServiceClient, ContentSettings

EMBEDDING_RUN = 'fine_keyframes_jina_clip_v2_1024d_v2'
AZURE_CONTAINER_EMBEDDINGS = 'embeddings'
AZURE_CONTAINER_KEYFRAMES = 'keyframes'
AZURE_KEYFRAMES_PREFIX = ''
OUTPUT_ROOT = Path('/kaggle/working') / f'{EMBEDDING_RUN}_merged_indexes'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_NAMES = ['jina']

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

connection_string = get_secret('AZURE_STORAGE_CONNECTION_STRING')
if not connection_string:
    raise RuntimeError('Missing Kaggle Secret AZURE_STORAGE_CONNECTION_STRING')
blob_service = BlobServiceClient.from_connection_string(connection_string)
container = blob_service.get_container_client(AZURE_CONTAINER_EMBEDDINGS)
keyframes_container = blob_service.get_container_client(AZURE_CONTAINER_KEYFRAMES)
container.get_container_properties()
keyframes_container.get_container_properties()

In [ ]:
# Reports are useful progress summaries, but a Kaggle session can stop after the
# last checkpoint and before writing its final report. Read them without requiring them.
reports = []
prefix = f'reports/{EMBEDDING_RUN}/'
for blob in container.list_blobs(name_starts_with=prefix):
    reports.append(json.loads(container.get_blob_client(blob.name).download_blob().readall()))
print('Reports found:', len(reports))
for report in sorted(reports, key=lambda item: item['part_name']):
    print(report['part_name'], 'completed=', report['completed'], 'total=', report['total'], 'failed=', len(report['failed']))
if not reports:
    print('No final reports found. Completion will be verified from keyframes + checkpoints.')
elif any(report.get('failed') for report in reports):
    print('WARNING: at least one report contains failures; checkpoint verification below is authoritative.')

In [ ]:
# Stream checkpoints, records, and NPY blobs. Vector IDs are assigned here, once.
import io
from collections import defaultdict

def download_json(blob_name):
    return json.loads(container.get_blob_client(blob_name).download_blob().readall())

def checkpoints():
    prefix = f'checkpoints/{EMBEDDING_RUN}/'
    items = []
    for blob in container.list_blobs(name_starts_with=prefix):
        item = download_json(blob.name)
        if item.get('status') == 'completed':
            items.append(item)
    return sorted(items, key=lambda x: (x['parent_namespace'] if 'parent_namespace' in x else x['namespace'], x['video_id']))

items = checkpoints()
keys = [(x['namespace'], x['video_id']) for x in items]
if len(keys) != len(set(keys)):
    raise RuntimeError('Duplicate video checkpoint(s) found; clean the Azure run before merge.')
print('Completed video checkpoints:', len(items))

JOB_PRESETS = {
    'part_01': ['L21_a', 'L22_a', 'L23_a'],
    'part_02': ['L24_a', 'L25_a', 'L26_a'],
    'part_03': ['L26_b', 'L26_c', 'L26_d'],
    'part_04': ['L26_e', 'L27_a', 'L28_a'],
    'part_05': ['L29_a', 'L30_a'],
}

def namespace_prefix(namespace):
    return '/'.join(
        value for value in (AZURE_KEYFRAMES_PREFIX.strip('/'), namespace)
        if value
    ) + '/'

expected_part_by_key = {}
for part_name, namespaces in JOB_PRESETS.items():
    for namespace in namespaces:
        prefix = namespace_prefix(namespace)
        video_ids = set()
        # Hierarchical listing returns one virtual directory per video without
        # downloading/listing every individual keyframe blob.
        for entry in keyframes_container.walk_blobs(name_starts_with=prefix, delimiter='/'):
            relative = entry.name[len(prefix):].strip('/')
            if relative:
                video_ids.add(relative.split('/')[0])
        print(part_name, namespace, 'expected videos=', len(video_ids))
        for video_id in video_ids:
            key = (namespace, video_id)
            if key in expected_part_by_key:
                raise RuntimeError(f'Duplicate expected video assignment: {key}')
            expected_part_by_key[key] = part_name

if not expected_part_by_key:
    raise RuntimeError('No videos discovered in the Azure keyframes container; check AZURE_KEYFRAMES_PREFIX.')

actual_by_key = {(item['namespace'], item['video_id']): item for item in items}
expected_keys = set(expected_part_by_key)
actual_keys = set(actual_by_key)
missing_keys = sorted(expected_keys - actual_keys)
extra_keys = sorted(actual_keys - expected_keys)
wrong_part = sorted(
    (key, expected_part_by_key[key], actual_by_key[key].get('part_name'))
    for key in expected_keys & actual_keys
    if actual_by_key[key].get('part_name') != expected_part_by_key[key]
)

print('Expected videos:', len(expected_keys))
print('Checkpoint videos:', len(actual_keys))
print('Missing:', len(missing_keys), '| Extra:', len(extra_keys), '| Wrong part:', len(wrong_part))
if missing_keys:
    print('Missing sample:', missing_keys[:20])
if extra_keys:
    print('Extra sample:', extra_keys[:20])
if wrong_part:
    print('Wrong-part sample:', wrong_part[:20])
if missing_keys or extra_keys or wrong_part:
    raise RuntimeError('Embedding checkpoints do not exactly cover the Azure keyframe videos. Rerun the affected workers.')

print('PASS: every Azure video has exactly one completed checkpoint in the correct part.')

In [ ]:
def build_model_index(model_name):
    model_root = OUTPUT_ROOT / model_name
    model_root.mkdir(parents=True, exist_ok=True)
    index = None
    records, video_rows, vector_id = [], [], 0

    for item_no, item in enumerate(items, 1):
        artifact = item['artifacts'].get(model_name)
        if not artifact:
            raise RuntimeError(f'{model_name} missing for {item["namespace"]}/{item["video_id"]}')
        record_payload = download_json(item['records_blob'])
        frame_records = record_payload['records']
        raw = container.get_blob_client(artifact['blob']).download_blob().readall()
        vectors = np.load(io.BytesIO(raw), allow_pickle=False).astype('float32', copy=False)
        if vectors.ndim != 2 or len(vectors) != len(frame_records) or not np.isfinite(vectors).all():
            raise RuntimeError(f'Invalid {model_name} artifact: {artifact["blob"]}')
        norms = np.linalg.norm(vectors, axis=1)
        if not np.allclose(norms, 1.0, atol=2e-3):
            raise RuntimeError(f'Non-normalized {model_name} artifact: {artifact["blob"]}')
        if index is None:
            index = faiss.IndexIDMap2(faiss.IndexFlatIP(int(vectors.shape[1])))
        elif index.d != vectors.shape[1]:
            raise RuntimeError(f'{model_name} dimension changed within run')
        ids = np.arange(vector_id, vector_id + len(vectors), dtype=np.int64)
        index.add_with_ids(vectors, ids)
        for offset, record in enumerate(frame_records):
            row = dict(record)
            row['vector_id'] = int(vector_id + offset)
            records.append(row)
        video_rows.append({
            'video_id': item['video_id'], 'parent_namespace': item['namespace'],
            'frame_count': len(vectors), 'embedding_dim': int(vectors.shape[1]),
            'first_vector_id': int(vector_id), 'artifact_blob': artifact['blob'],
        })
        vector_id += len(vectors)
        if item_no % 25 == 0:
            print(model_name, item_no, '/', len(items), 'videos;', vector_id, 'vectors')

    if index is None or index.ntotal != len(records):
        raise RuntimeError(f'{model_name}: index/record mismatch')
    global_ids = pd.DataFrame.from_records(records)
    if global_ids['vector_id'].tolist() != list(range(len(global_ids))):
        raise RuntimeError(f'{model_name}: non-contiguous vector IDs')
    global_ids.to_parquet(model_root / 'global_ids.parquet', index=False)
    pd.DataFrame.from_records(video_rows).to_parquet(model_root / 'video_metadata.parquet', index=False)
    faiss.write_index(index, str(model_root / f'{model_name}_faiss.index'))
    meta = {
        'embedding_run': EMBEDDING_RUN, 'model': model_name, 'embedding_dim': int(index.d),
        'metric': 'inner_product_on_l2_normalized_vectors', 'vector_count': int(index.ntotal),
        'video_count': len(video_rows), 'source': 'Azure per-video artifacts',
    }
    (model_root / 'index_meta.json').write_text(json.dumps(meta, indent=2) + '\n', encoding='utf-8')
    print('Built', model_name, 'vectors=', index.ntotal, 'dim=', index.d)
    return model_root, meta

built = [build_model_index(model_name) for model_name in MODEL_NAMES]

In [ ]:
# Validate locally, then publish final artifacts to Azure under a versioned prefix.
import hashlib

def sha256_path(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

for model_root, meta in built:
    model_name = meta['model']
    index_path = model_root / f'{model_name}_faiss.index'
    loaded_index = faiss.read_index(str(index_path))
    ids = pd.read_parquet(model_root / 'global_ids.parquet')
    assert loaded_index.ntotal == len(ids) == meta['vector_count']
    assert loaded_index.d == meta['embedding_dim']
    assert ids['vector_id'].is_unique
    assert ids['vector_id'].tolist() == list(range(len(ids)))
    assert ids[['parent_namespace', 'video_id', 'frame_path']].duplicated().sum() == 0
    print('PASS:', model_name, meta)

    for path in model_root.iterdir():
        remote = f'indexes/{EMBEDDING_RUN}/{model_name}/{path.name}'
        with path.open('rb') as handle:
            container.get_blob_client(remote).upload_blob(
                handle, overwrite=True, metadata={'sha256': sha256_path(path)},
                content_settings=ContentSettings(content_type='application/octet-stream'),
            )
        print('Uploaded:', remote)

print('\nJina artifacts for the matching backend retriever:')
print('JINA_FAISS_INDEX_PATH=.../jina_faiss.index')
print('JINA_GLOBAL_IDS_PATH=.../global_ids.parquet')
print('JINA_VIDEO_METADATA_PATH=.../video_metadata.parquet')
print('JINA_INDEX_META_PATH=.../index_meta.json')